In [130]:
from data_pipeline import *
from environment import *
from movement import *


import numpy as np
import pandas as pd

from tqdm import tqdm

import pickle
import math

In [ ]:
with open("Q.pkl", "rb") as f:
    Q = pickle.load(f)

In [131]:
def get_actions(combined_action):
    action1 = math.floor(combined_action / 4)
    action2 = combined_action % 4
    return All_Actions(action1, action2)


In [132]:
Q = {}

all_positions = []
all_rewards = []
env = GridWorld_Portal()

Q[env.all_entity_positions] = [200] * 16

In [133]:
def epsilon_greedy(state, epsilon):

    if np.random.random() < epsilon:
        return np.random.randint(12)
    else:
        combined_a = np.argmax(Q[state]).item()
        return combined_a

In [134]:
def update_Q(previous_state, action, next_state, reward):

    if next_state not in Q:
        Q[next_state] = [200] * 16

    Q[previous_state][action] = Q[previous_state][action] + 0.1 * (
        reward + 0.99 * max(Q[next_state]) - Q[previous_state][action]
    )

In [135]:
def get_reward(previous_state, next_state):

    reward = 0
    _, _, box_pos = previous_state.get_positions()
    next_pos1, next_pos2, next_box_pos = next_state.get_positions()

    if next_state not in Q:
        reward += 1


    if box_pos != next_box_pos:
        reward += 2

    if next_pos1.get_position()[0] > 9:
        reward += 3

    if next_pos2.get_position()[0] > 9:
        reward += 3

    return reward

In [137]:
for i in tqdm(range(30000)):

    env.reset()
    rewards = [0] * 1000
    positions = [0] * 1000

    for j in range(1000):
        previous_state = env.all_entity_positions
        positions[j] = previous_state

        combined_action = epsilon_greedy(previous_state, 0.2)
        actions = get_actions(combined_action)

        next_state = env.step(actions)

        reward = get_reward(previous_state, next_state)

        rewards[j] = reward

        update_Q(previous_state, combined_action, next_state, reward)



    all_positions.append(positions)
    all_rewards.append(rewards)


  6%|▌         | 1733/30000 [01:13<19:52, 23.71it/s]


KeyboardInterrupt: 

In [76]:
def show_value_counts(all_rewards):
    for rewards in all_rewards:
        print(pd.Series(rewards).value_counts())

In [129]:
# Save
with open("Q.pkl", "wb") as f:
    pickle.dump(Q, f)